# 03 — Data Cleaning

The raw dataset intentionally contains missing values, inconsistent text, and duplicated rows. Keep the raw data unchanged.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
raw = pd.read_csv(ROOT / 'data' / 'raw' / 'sales.csv')
df = raw.copy()
print('Rows:', len(df), 'Duplicates:', df.duplicated().sum())
print(df.isna().sum())

## Standardize column names and text

In [ ]:
df.columns = df.columns.str.strip()
for col in ['Product', 'Category', 'Region', 'Payment_Method', 'Customer_ID']:
    df[col] = df[col].astype('string').str.strip()
df['Region'] = df['Region'].str.title()
df['Product'] = df['Product'].str.title()
print(df['Region'].value_counts(dropna=False))

## Convert types and inspect missingness

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
for col in ['Quantity', 'Unit_Price', 'Discount']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print(df.dtypes)
print(df.isna().sum())

## Choose and document cleaning rules

In [ ]:
df = df.drop_duplicates()
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())
df['Unit_Price'] = df['Unit_Price'].fillna(df.groupby('Product')['Unit_Price'].transform('median'))
df['Discount'] = df['Discount'].fillna(0)
df['Region'] = df['Region'].fillna('Unknown')
df = df.dropna(subset=['Order_ID', 'Order_Date', 'Product', 'Customer_ID'])
df['Quantity'] = df['Quantity'].round().astype('int64')
df['Discount'] = df['Discount'].clip(0, 1)
df['Net_Sales'] = df['Quantity'] * df['Unit_Price'] * (1 - df['Discount'])
print('Cleaned shape:', df.shape)
print(df.isna().sum())

## Save and validate

In [ ]:
out = ROOT / 'data' / 'processed'
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / 'sales_cleaned.csv', index=False)
assert df['Order_ID'].is_unique, 'Check whether Order_ID is expected to be unique after duplicate removal'
assert (df['Quantity'] > 0).all()
assert df['Discount'].between(0, 1).all()
print('Saved cleaned data.')

## Exercises
1. Compare missing-value counts before and after cleaning.
2. Try median vs product-specific median imputation and explain the tradeoff.
3. Find records with non-positive prices or quantities before deciding whether to remove them.
4. Document every cleaning rule and its reason in your README.